# Next step: step 4c, `SimulatorConfig`
**What it adds:** one object for everything that sets a run's result, apart from the 22 parameters and the
seed. `simulate` and `initial_state` take it instead of separate keyword arguments.

**Why:**
- **`simr` will save the exact config of every run.** `asdict(config)` gives every value, defaults included, in
  one line, ready to save as JSON. It's the `simulator` section of the config tree in `PLAN.md`.
- **Checks happen once, when the config is made**, before any run starts, instead of partway through a long job.
- **`simulate(params, config, seed)` stays short** as the mass model (step 5) and endings (step 7) add fields.

**Why `seed` isn't in it:** the seed belongs to the model, not the simulator. Later, `simr` gives every
simulator run on one model the same seed, so differences between them don't come from different particles.

**Not in it:** `potential`. `sim` always uses SPAM's extended potential (halo, disk and bulge); `jspam` still
has the switch if that's ever needed.

No change to JSPAM in this step. The full, current demo is `simulator_tour.ipynb`.

**To run it** (from this repo's root, in bash or PowerShell), then open the `127.0.0.1` link it prints:
```bash
docker run --rm -it -p 8888:8888 -v "${PWD}:/gtc" -w /gtc -e UV_PROJECT_ENVIRONMENT=/tmp/venv \
    ogdenm12/gtc-base-image:v2 uv run --group notebook jupyter lab --ip=0.0.0.0 --allow-root --no-browser
```
- `-v "${PWD}:/gtc"` mounts this repo into the container, so edits on Windows show up immediately.
- `UV_PROJECT_ENVIRONMENT=/tmp/venv` keeps the Linux virtual environment inside the container, out of the repo.
  It's rebuilt each time the container starts (a minute or two, mostly compiling SPAM).
- Our package is installed in editable mode. After editing a `.py` file, restart the kernel to pick it up.

## What changed in `sim.py`
**Added:** the dataclass. `@dataclass(frozen=True)` writes the constructor, the printout and `==` from the field
list, and makes it unchangeable after it's made. `__post_init__` runs right after the constructor.
```python
@dataclass(frozen=True)
class SimulatorConfig:
    n1: int = 1000
    n2: int = 1000
    heat1: float = 0.0
    heat2: float = 0.0
    heat_mode: str = "centered"

    def __post_init__(self) -> None:
        for name in ("n1", "n2"):
            value = getattr(self, name)
            if not isinstance(value, (int, np.integer)) or value < 1:
                raise ValueError(f"{name} must be a whole number of at least 1: {value!r}")
        for name in ("heat1", "heat2"):
            if getattr(self, name) < 0:
                raise ValueError(f"{name} can't be negative: {getattr(self, name)!r}")
        if self.heat_mode not in jspam.HEAT_MODES:
            raise ValueError(f"heat_mode must be one of {list(jspam.HEAT_MODES)}: {self.heat_mode!r}")
```
**Changed:** `simulate` and `initial_state` take a config, and a small helper turns it into `jspam`'s keyword
arguments.
```python
# before
def simulate(params, n1=1000, n2=1000, seed=None, heat1=0.0, heat2=0.0, heat_mode="centered"): ...
# after
def simulate(params, config=SimulatorConfig(), seed=None):
    return jspam.run_collision(params, seed=seed, **_jspam_args(config))


def _jspam_args(config):
    return {"n1": config.n1, "n2": config.n2, "heat1": config.heat1, "heat2": config.heat2,
            "heat_mode": config.heat_mode}
```
**`tests/test_sim.py`:** every test now builds a `SimulatorConfig`, and a new section checks the config itself.

In [ ]:
from dataclasses import FrozenInstanceError, asdict, replace

import numpy as np

from galactic_tidal_constraints.simr import sim

MODEL_STRING = "-9.93853,-4.5805,15.43348,-1.54025,-2.97899,4.69234,34.41671,29.49607,2.97913,4.2501,301.0,35.5,310.80745,321.98757,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0.24008,-2.41999,5.62715,0.0,1.0,0.0,1.0,0.0,0.0"
params = sim.parse_model_string(MODEL_STRING)

## 1. Making a config
Leave out a field and it takes its default. The printout shows every field.

In [ ]:
default = sim.SimulatorConfig()
small_hot = sim.SimulatorConfig(n1=300, n2=200, heat1=0.5, heat2=0.5)
print(default)
print(small_hot)

## 2. Every value, ready to save
`asdict` includes the defaults, so a saved config says exactly how a run was simulated, even for values nobody
set. Going back is `SimulatorConfig(**the_dict)`.

In [ ]:
saved = asdict(small_hot)
print(saved)
loaded = sim.SimulatorConfig(**saved)
print("same config after the round trip:", loaded == small_hot)

## 3. Changing a value
Frozen: no changes in place, so a config passed to many runs can't be altered by one of them. `replace` makes a
changed copy.

In [ ]:
try:
    small_hot.n1 = 5
except FrozenInstanceError as error:
    print("can't change it in place:", error)

wallin = replace(small_hot, heat_mode="wallin")
print(wallin)

## 4. Mistakes are caught when the config is made
Before any simulation runs. A field that doesn't exist, like `potential`, is caught too.

In [ ]:
for bad in [{"n1": 0}, {"n1": 10.5}, {"heat1": -0.1}, {"heat_mode": "centred"}, {"potential": "point_mass"}]:
    try:
        sim.SimulatorConfig(**bad)
    except (ValueError, TypeError) as error:
        print(f"{bad}  ->  {type(error).__name__}: {error}")

## 5. Running with a config
The same seed with the same config gives identical particles. Two configs that differ only in heat mode, with
no heat, give the same run.

In [ ]:
a = sim.simulate(params, small_hot, seed=1)
b = sim.simulate(params, sim.SimulatorConfig(**saved), seed=1)
print("shape:", a.shape)
print("loaded config gives the identical run:", np.array_equal(a, b))

no_heat = sim.SimulatorConfig(n1=300, n2=200)
c = sim.simulate(params, no_heat, seed=1)
d = sim.simulate(params, replace(no_heat, heat_mode="wallin"), seed=1)
print("no heat, both modes identical:", np.array_equal(c, d))

## 6. One model, several simulators
`PLAN.md` lets a stage have a list of configs. Here, one model and one seed, run with three configs. The seed is
shared, so the differences come only from the configs.

In [ ]:
configs = [
    sim.SimulatorConfig(n1=500, n2=500),
    sim.SimulatorConfig(n1=500, n2=500, heat1=0.3, heat2=0.3),
    sim.SimulatorConfig(n1=500, n2=500, heat1=0.3, heat2=0.3, heat_mode="wallin"),
]
for config in configs:
    final = sim.simulate(params, config, seed=1)
    print(f"{asdict(config)}   primary disk's mean position: {final[:config.n1, :3].mean(axis=0).round(3)}")